In [1]:
import os
from datetime import datetime
from io import StringIO

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
os.makedirs("data/raw", exist_ok=True)
import re


In [2]:
# PxAPI v1 POST endpoint for CJA07
url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.PxAPIv1/en/20/RC/CJA07"

# For full table: empty query + CSV response
payload = {
    "query": [],
    "response": {
        "format": "csv",
        "pivot": None,
        "codes": False
    }
}

response = requests.post(url, json=payload)
response.raise_for_status()  # will raise an error if something is wrong

csv_text = response.text
df = pd.read_csv(StringIO(csv_text))

print('Data Loaded Successfully. \n')

df.head()


Data Loaded Successfully. 



,"ï»¿""STATISTIC Label""",Year,Garda Station,Type of Offence,UNIT,VALUE
0,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division","Attempts/threats to murder, assaults, harassme...",Number,18
1,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division",Dangerous or negligent acts (04),Number,27
2,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division",Kidnapping and related offences (05),Number,0
3,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division","Robbery, extortion and hijacking offences (06)",Number,0
4,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division",Burglary and related offences (07),Number,27


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173712 entries, 0 to 173711
Data columns (total 6 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   ï»¿"STATISTIC Label"  173712 non-null  object
 1   Year                  173712 non-null  int64 
 2   Garda Station         173712 non-null  object
 3   Type of Offence       173712 non-null  object
 4   UNIT                  173712 non-null  object
 5   VALUE                 173712 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 8.0+ MB


In [4]:
# check for duplicates

duplicates = df.duplicated().sum()

print('No.Of Duplicates in the dataframe : ', duplicates)

key_cols = ["Year", "Garda Station", "Type of Offence"]
dup_keys = df.duplicated(subset=key_cols).sum()

print(f"\n Duplicate rows on key {key_cols}: {dup_keys}")
if dup_keys > 0:
    print(" There are duplicates at the Year×Station×Offence level. We'll inspect samples below.")
    display(df[df.duplicated(subset=key_cols, keep=False)].head(10))
else:
    print(" No duplicates at the Year × Station × Offence level.")

No.Of Duplicates in the dataframe :  0

 Duplicate rows on key ['Year', 'Garda Station', 'Type of Offence']: 0
 No duplicates at the Year × Station × Offence level.


In [5]:
nulls = df.isnull().sum()

print('No.Of Nulls in the dataframe : \n ', nulls)

No.Of Nulls in the dataframe : 
  ï»¿"STATISTIC Label"    0
Year                    0
Garda Station           0
Type of Offence         0
UNIT                    0
VALUE                   0
dtype: int64


In [6]:
# Check unique value counts for each column
for column in df.columns:
    print(f"Number of unique values in '{column}': {df[column].nunique()}\n")

Number of unique values in 'ï»¿"STATISTIC Label"': 1

Number of unique values in 'Year': 22

Number of unique values in 'Garda Station': 564

Number of unique values in 'Type of Offence': 14

Number of unique values in 'UNIT': 1

Number of unique values in 'VALUE': 1592



In [7]:
# rename columns
df = df.rename(columns={'ï»¿"STATISTIC Label"': 'Statistic Label'})
df = df.rename(columns={'VALUE' : 'Value'})

print('columns renamed : \n ', df[['Statistic Label', 'Value']])

columns renamed : 
                   Statistic Label  Value
0       Recorded crime incidents     18
1       Recorded crime incidents     27
2       Recorded crime incidents      0
3       Recorded crime incidents      0
4       Recorded crime incidents     27
...                          ...    ...
173707  Recorded crime incidents     39
173708  Recorded crime incidents     62
173709  Recorded crime incidents     42
173710  Recorded crime incidents     18
173711  Recorded crime incidents      4

[173712 rows x 2 columns]


In [8]:
# Station and offence group counts
# How many unique stations and offence types are we working with?

print("STATION & OFFENCE GROUP COUNTS \n")


print(f"Number of unique Garda Stations: {df['Garda Station'].nunique()}")
print(f"Number of unique Type of Offence: {df['Type of Offence'].nunique()}")

print("\nSample station names:")
print(df['Garda Station'].drop_duplicates().head(8).tolist())

print("\nSample offence types:")
print(df['Type of Offence'].drop_duplicates().head(8).tolist())

print(f"\nStations with most incidents:")
top_stations = df.groupby('Garda Station')['Value'].sum().nlargest(5)
print(top_stations)

STATION & OFFENCE GROUP COUNTS 

Number of unique Garda Stations: 564
Number of unique Type of Offence: 14

Sample station names:
['35301 Abbeyfeale, Limerick Division', '41201 Abbeyleix, Laois/Offaly Division', '35302 Adare, Limerick Division', '54101 Aglish, Waterford Division', '23101 Ahascragh, Galway Division', '12501 Ailt an Chorraiin, Donegal Division', '12502 An Bun Beag, Donegal Division', '12503 An Charraig, Donegal Division']

Sample offence types:
['Attempts/threats to murder, assaults, harassments and related offences (03)', 'Dangerous or negligent acts (04)', 'Kidnapping and related offences (05)', 'Robbery, extortion and hijacking offences (06)', 'Burglary and related offences (07)', 'Theft and related offences (08)', 'Fraud, deception and related offences (09)', 'Controlled drug offences (10)']

Stations with most incidents:
Garda Station
61202 Pearse Street, D.M.R. South Central Division       258096
62301 Store Street, D.M.R. North Central Division        257130
64202

In [9]:
value_stats = df['Value'].describe()

print('Statistics of the value feature : \n ', value_stats)

Statistics of the value feature : 
  count    173712.000000
mean         44.783774
std         171.197615
min           0.000000
25%           1.000000
50%           5.000000
75%          23.000000
max        6524.000000
Name: Value, dtype: float64


In [10]:
# column integrity checks for non-standard columns
checks = [
    {"column": "UNIT", "label": "Units", "operation": lambda col: df[col].unique()},
    {"column": "Statistic Label", "label": "Statistics", "operation": lambda col: df[col].unique()}
]

for check in checks:
    try:
        result = check["operation"](check["column"])
        print(f"{check['label']}: {result}")
    except KeyError:
        print(f"Warning: '{check['column']}' column not found.")

Units: ['Number']
Statistics: ['Recorded crime incidents']


In [11]:
df.head()

,Statistic Label,Year,Garda Station,Type of Offence,UNIT,Value
0,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division","Attempts/threats to murder, assaults, harassme...",Number,18
1,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division",Dangerous or negligent acts (04),Number,27
2,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division",Kidnapping and related offences (05),Number,0
3,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division","Robbery, extortion and hijacking offences (06)",Number,0
4,Recorded crime incidents,2003,"35301 Abbeyfeale, Limerick Division",Burglary and related offences (07),Number,27


In [12]:
# check statistic label and unit so that we can drop
print(f'The Unique values in Statistic Labels Column : {df['Statistic Label'].unique()}')
print(f'The Unique Values in Unit Column : {df['UNIT'].unique()}')


The Unique values in Statistic Labels Column : ['Recorded crime incidents']
The Unique Values in Unit Column : ['Number']


In [13]:
# since both columns have only 1 value, we removing them
df = df.drop(['Statistic Label', 'UNIT'], axis=1)

print('Statistic Label and Unit columns dropped successfully. ')

Statistic Label and Unit columns dropped successfully. 


In [14]:
df.head()

,Year,Garda Station,Type of Offence,Value
0,2003,"35301 Abbeyfeale, Limerick Division","Attempts/threats to murder, assaults, harassme...",18
1,2003,"35301 Abbeyfeale, Limerick Division",Dangerous or negligent acts (04),27
2,2003,"35301 Abbeyfeale, Limerick Division",Kidnapping and related offences (05),0
3,2003,"35301 Abbeyfeale, Limerick Division","Robbery, extortion and hijacking offences (06)",0
4,2003,"35301 Abbeyfeale, Limerick Division",Burglary and related offences (07),27


In [15]:
# Missing values summary
print("MISSING VALUES SUMMARY \n")


missing_summary = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_summary,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

print(missing_df)
print(f"\nTotal rows with any missing values: {df.isnull().any(axis=1).sum()}")


MISSING VALUES SUMMARY 

                 missing_count  missing_pct
Year                         0          0.0
Garda Station                0          0.0
Type of Offence              0          0.0
Value                        0          0.0

Total rows with any missing values: 0


In [16]:
# validation
years = df["Year"].nunique()
stations = df["Garda Station"].nunique()
offences = df["Type of Offence"].nunique()

print("Expected grid:", years * stations * offences)
print("Actual rows:", len(df))

Expected grid: 173712
Actual rows: 173712


In [17]:
# Calculate the proportion of 'Value' entries that are 0.

zero_value_proportion = (df["Value"] == 0).mean()
print(f"Proportion of records with zero 'Value': {zero_value_proportion:.4f}")

Proportion of records with zero 'Value': 0.2238


In [18]:
# Calculate the coverage for each Garda Station over the years.

coverage = (
    df.groupby("Garda Station")["Year"]
      .agg(["min", "max", "nunique"])
      .reset_index()
)


print("Coverage statistics for Garda Stations:")
coverage.head()

Coverage statistics for Garda Stations:


,Garda Station,min,max,nunique
0,"11101 Bailieboro, Cavan/Monaghan Division",2003,2024,22
1,"11102 Ballyjamesduff, Cavan/Monaghan Division",2003,2024,22
2,"11103 Cootehill, Cavan/Monaghan Division",2003,2024,22
3,"11104 Kingscourt, Cavan/Monaghan Division",2003,2024,22
4,"11105 Mullagh, Cavan/Monaghan Division",2003,2024,22


In [19]:
# count of total incidents per year

print("National total incidents by year :")
national_by_year = (
    df.groupby("Year")["Value"]
      .sum()
      .reset_index()
      .sort_values("Year")
)
display(national_by_year.head(25))

National total incidents by year :


,Year,Value
0,2003,283913
1,2004,298319
2,2005,335947
3,2006,445102
4,2007,488299
5,2008,511461
6,2009,472400
7,2010,439725
8,2011,418286
9,2012,384337


In [20]:
# copy datframe so you don't modify the raw df accidentally
df_clean = df.copy()

# Split Garda Station
df_clean["station_code"] = df_clean["Garda Station"].str.extract(r"^(\d+)")[0]
df_clean["station_name"] = df_clean["Garda Station"].str.extract(r"^\d+\s+([^,]+)")[0].str.strip()
df_clean["station_division"] = df_clean["Garda Station"].str.extract(r",(.+)$")[0].str.strip()

# remove " Division" suffix for cleaner grouping/plots
df_clean["station_division"] = df_clean["station_division"].str.replace(" Division", "", regex=False)

# Split Type of Offence
df_clean["offence_group_name"] = df_clean["Type of Offence"].str.extract(r"^(.+)\s+\(\d+\)$")[0].str.strip()
df_clean["offence_group_code"] = df_clean["Type of Offence"].str.extract(r"\((\d+)\)$")[0]

print("Parsing completed.")
print("Sample rows:")
df_clean.head()


Parsing completed.
Sample rows:


,Year,Garda Station,Type of Offence,Value,station_code,station_name,station_division,offence_group_name,offence_group_code
0,2003,"35301 Abbeyfeale, Limerick Division","Attempts/threats to murder, assaults, harassme...",18,35301,Abbeyfeale,Limerick,"Attempts/threats to murder, assaults, harassme...",03
1,2003,"35301 Abbeyfeale, Limerick Division",Dangerous or negligent acts (04),27,35301,Abbeyfeale,Limerick,Dangerous or negligent acts,04
2,2003,"35301 Abbeyfeale, Limerick Division",Kidnapping and related offences (05),0,35301,Abbeyfeale,Limerick,Kidnapping and related offences,05
3,2003,"35301 Abbeyfeale, Limerick Division","Robbery, extortion and hijacking offences (06)",0,35301,Abbeyfeale,Limerick,"Robbery, extortion and hijacking offences",06
4,2003,"35301 Abbeyfeale, Limerick Division",Burglary and related offences (07),27,35301,Abbeyfeale,Limerick,Burglary and related offences,07


In [21]:
# Basic validation prints
print("Missing station_code:", df_clean["station_code"].isna().sum())
print("Missing station_name:", df_clean["station_name"].isna().sum())
print("Missing station_division:", df_clean["station_division"].isna().sum())
print("Missing offence_group_code:", df_clean["offence_group_code"].isna().sum())
print("Missing offence_group_name:", df_clean["offence_group_name"].isna().sum())

Missing station_code: 0
Missing station_name: 0
Missing station_division: 0
Missing offence_group_code: 0
Missing offence_group_name: 0


In [22]:
# Validation
print("Unique stations before/after split:")
print("Garda Station:", df['Garda Station'].nunique())
print("station_code:", df_clean['station_code'].nunique())

print("\nUnique offences before/after split:")
print("Type of Offence:", df['Type of Offence'].nunique())
print("offence_group_code:", df_clean['offence_group_code'].nunique())


Unique stations before/after split:
Garda Station: 564
station_code: 564

Unique offences before/after split:
Type of Offence: 14
offence_group_code: 14


In [23]:
print("Sample splits (unique rows):")
cols_show = ["Garda Station", "station_code", "station_name", "station_division",
             "Type of Offence", "offence_group_name", "offence_group_code"]
display(df_clean[cols_show].drop_duplicates().head(12))

# Quick stability check
division_check = (
    df_clean.groupby("station_code")["station_division"]
    .nunique()
    .sort_values(ascending=False)
)

multi_div = division_check[division_check > 1]

print("\nStations mapped to multiple divisions (if any):", len(multi_div))
if len(multi_div) > 0:
    print(multi_div.head(15))


Sample splits (unique rows):


,Garda Station,station_code,station_name,station_division,Type of Offence,offence_group_name,offence_group_code
0,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,"Attempts/threats to murder, assaults, harassme...","Attempts/threats to murder, assaults, harassme...",03
1,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,Dangerous or negligent acts (04),Dangerous or negligent acts,04
2,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,Kidnapping and related offences (05),Kidnapping and related offences,05
3,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,"Robbery, extortion and hijacking offences (06)","Robbery, extortion and hijacking offences",06
4,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,Burglary and related offences (07),Burglary and related offences,07
5,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,Theft and related offences (08),Theft and related offences,08
6,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,"Fraud, deception and related offences (09)","Fraud, deception and related offences",09
7,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,Controlled drug offences (10),Controlled drug offences,10
8,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,Weapons and explosives offences (11),Weapons and explosives offences,11
9,"35301 Abbeyfeale, Limerick Division",35301,Abbeyfeale,Limerick,Damage to property and to the environment off...,Damage to property and to the environment off...,12



Stations mapped to multiple divisions (if any): 0


In [24]:
# top 10 stations and offences

print("Top 10 stations by total incidents (all years, all offences):")
top_stations = (
    df_clean.groupby("station_name")["Value"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)
print(top_stations)

print("\nTop 10 offence groups by total incidents (all years, all stations):")
top_offences = (
    df_clean.groupby("offence_group_name")["Value"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)
print(top_offences)

Top 10 stations by total incidents (all years, all offences):
station_name
Pearse Street       258096
Store Street        257130
Tallaght            185904
Blanchardstown      177488
Bridewell Dublin    164239
Henry Street        137085
Gaillimh            136959
Waterford           128771
Clondalkin          101880
Kevin Street         98747
Name: Value, dtype: int64

Top 10 offence groups by total incidents (all years, all stations):
offence_group_name
Road and traffic offences                                                    1880513
Theft and related offences                                                   1566268
Public order and other social code offences                                   924455
Dangerous or negligent acts                                                   915139
Damage to property and to the environment  offences                           665813
Burglary and related offences                                                 461603
Attempts/threats to murder, ass

In [26]:
df_clean.head()

,Year,Garda Station,Type of Offence,Value,station_code,station_name,station_division,offence_group_name,offence_group_code
0,2003,"35301 Abbeyfeale, Limerick Division","Attempts/threats to murder, assaults, harassme...",18,35301,Abbeyfeale,Limerick,"Attempts/threats to murder, assaults, harassme...",03
1,2003,"35301 Abbeyfeale, Limerick Division",Dangerous or negligent acts (04),27,35301,Abbeyfeale,Limerick,Dangerous or negligent acts,04
2,2003,"35301 Abbeyfeale, Limerick Division",Kidnapping and related offences (05),0,35301,Abbeyfeale,Limerick,Kidnapping and related offences,05
3,2003,"35301 Abbeyfeale, Limerick Division","Robbery, extortion and hijacking offences (06)",0,35301,Abbeyfeale,Limerick,"Robbery, extortion and hijacking offences",06
4,2003,"35301 Abbeyfeale, Limerick Division",Burglary and related offences (07),27,35301,Abbeyfeale,Limerick,Burglary and related offences,07


In [25]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173712 entries, 0 to 173711
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   Year                173712 non-null  int64 
 1   Garda Station       173712 non-null  object
 2   Type of Offence     173712 non-null  object
 3   Value               173712 non-null  int64 
 4   station_code        173712 non-null  object
 5   station_name        173712 non-null  object
 6   station_division    173712 non-null  object
 7   offence_group_name  173712 non-null  object
 8   offence_group_code  173712 non-null  object
dtypes: int64(2), object(7)
memory usage: 11.9+ MB


In [27]:
# Build final dataset

df_final = df_clean.copy()

# Rename columns properly
df_final = df_final.rename(columns={
    "Year": "year",
    "Value": "incidents"
})

# Keep only structured columns
df_final = df_final[[
    "year",
    "station_code",
    "station_name",
    "station_division",
    "offence_group_code",
    "offence_group_name",
    "incidents"
]].copy()

print("Final dataset shape:", df_final.shape)
print("Columns:", list(df_final.columns))

Final dataset shape: (173712, 7)
Columns: ['year', 'station_code', 'station_name', 'station_division', 'offence_group_code', 'offence_group_name', 'incidents']


In [35]:
df_final["year"] = pd.to_numeric(df_final["year"], errors="coerce")
df_final["incidents"] = pd.to_numeric(df_final["incidents"], errors="coerce")

print("Dataframe after dropping columns: \n")
df_final.info()

Dataframe after dropping columns: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173712 entries, 0 to 173711
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   year                173712 non-null  int64 
 1   station_code        173712 non-null  object
 2   station_name        173712 non-null  object
 3   station_division    173712 non-null  object
 4   offence_group_code  173712 non-null  object
 5   offence_group_name  173712 non-null  object
 6   incidents           173712 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 9.3+ MB


In [36]:
df_final.head()

,year,station_code,station_name,station_division,offence_group_code,offence_group_name,incidents
0,2003,35301,Abbeyfeale,Limerick,03,"Attempts/threats to murder, assaults, harassme...",18
1,2003,35301,Abbeyfeale,Limerick,04,Dangerous or negligent acts,27
2,2003,35301,Abbeyfeale,Limerick,05,Kidnapping and related offences,0
3,2003,35301,Abbeyfeale,Limerick,06,"Robbery, extortion and hijacking offences",0
4,2003,35301,Abbeyfeale,Limerick,07,Burglary and related offences,27


In [38]:
# Save locally in Colab environment
csv_path = "ireland_crime_structured_dataset.csv"

df_final.to_csv(csv_path, index=False)

print("\nFile saved successfully:")
print(" -", csv_path)

# Provide a link to download the file to the local machine
from google.colab import files
files.download(csv_path)


File saved successfully:
 - ireland_crime_structured_dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>